In [1]:
!pip install pandas

  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl.metadata (19 kB)
Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)

   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   ------------

In [9]:
import os
import time
import shutil
import cv2
import torch
import pandas as pd
from ultralytics import YOLO

In [10]:
from IPython import display
display.clear_output()
from IPython.display import display, Image

In [11]:
!nvidia-smi

Mon Apr 27 23:04:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.29                 Driver Version: 581.29         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2050      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   53C    P8              2W /   70W |     831MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version: 12.6
GPU count: 1
GPU name: NVIDIA GeForce RTX 2050


In [13]:
DATA = "dataset/Dataset KRSRI 2024 -2/data.yaml"

TEST_IMAGE = "test.jpg"

RUN_DIR = "./benchmark_runs"
EXPORT_DIR = "./trained_models"

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

RESULTS=[]

In [14]:
def train_benchmark(model_path):

    print("\n"+"="*70)
    print("TRAINING:",model_path)
    print("="*70)

    model_name=model_path.replace(".pt","")


    ##############################
    # MODEL-SPECIFIC BATCH
    ##############################

    batch_size=8
    if model_path in ["yolov6n.yaml","pretrained_models/yolov7.pt"]:
        batch_size=4


    ##############################
    # LOAD PRETRAINED MODEL
    ##############################

    model=YOLO(model_path)


    ##############################
    # TRAIN
    ##############################

    start=time.time()

    model.train(
        data=DATA,
        epochs=80,
        imgsz=640,
        batch=batch_size,
        patience=5,
        save=True,
        device=0,
        amp=True,

        optimizer="AdamW",
        close_mosaic=10,

        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0007,

        multi_scale=0.25,

        plots=True,
        seed=42,
        deterministic=True,

        project=RUN_DIR,
        name=model_name,
        exist_ok=False
    )

    train_minutes=(time.time()-start)/60


    ##############################
    # LOAD BEST TRAINED WEIGHT
    ##############################

    best_weight=str(model.trainer.best)

    print("Best weight:",best_weight)

    trained_model=YOLO(best_weight)


    ##############################
    # VALIDATION
    ##############################

    metrics=trained_model.val(
        split="val"
    )

    map50=float(metrics.box.map50)
    map5095=float(metrics.box.map)

    precision=float(metrics.box.mp)
    recall=float(metrics.box.mr)


    ##############################
    # MODEL SIZE
    ##############################

    size_mb=os.path.getsize(best_weight)/(1024**2)


    ####################################################
    # END-TO-END FPS (real deployment)
    ####################################################

    img=cv2.imread(TEST_IMAGE)

    for _ in range(20):
        trained_model.predict(
            img,
            device=0,
            verbose=False
        )

    torch.cuda.synchronize()

    N=100

    t1=time.time()

    for _ in range(N):
        trained_model.predict(
            img,
            device=0,
            verbose=False
        )

    torch.cuda.synchronize()

    t2=time.time()

    e2e_elapsed=t2-t1

    fps_e2e=N/e2e_elapsed
    latency_ms=(e2e_elapsed/N)*1000


    ####################################################
    # RAW MODEL FORWARD FPS
    ####################################################

    net=trained_model.model.cuda()
    net.eval()

    dummy=torch.randn(
        1,3,640,640,
        device="cuda"
    )

    for _ in range(50):
        _=net(dummy)

    torch.cuda.synchronize()

    N2=300

    t1=time.time()

    with torch.no_grad():
        for _ in range(N2):
            _=net(dummy)

    torch.cuda.synchronize()

    t2=time.time()

    raw_elapsed=t2-t1

    fps_raw=N2/raw_elapsed


    ##############################
    # COPY MODEL
    ##############################

    exported_model=(
        f"{EXPORT_DIR}/{model_name}_best.pt"
    )

    shutil.copy(
        best_weight,
        exported_model
    )


    ##############################
    # SAVE RESULTS
    ##############################

    RESULTS.append({
        "Model":model_name,

        "mAP50":map50,
        "mAP50-95":map5095,

        "Precision":precision,
        "Recall":recall,

        "FPS_E2E":fps_e2e,
        "FPS_Raw":fps_raw,

        "Latency_ms":latency_ms,

        "Size_MB":size_mb,

        "Training_Min":train_minutes
    })


    ##############################
    # PRINT
    ##############################

    print("\nDONE:",model_name)
    print("mAP50:",map50)
    print("E2E FPS:",fps_e2e)
    print("RAW FPS:",fps_raw)
    print("Latency(ms):",latency_ms)

In [15]:
# Jenis-jenis Yolo yang akan di benchmark
MODELS = [
    "yolov3-tinyu.pt",
    "yolov6n.yaml",
    "yolov10n.pt"
]

In [16]:
for i in range(len(MODELS)):
    train_benchmark(MODELS[i])


TRAINING: yolov3-tinyu.pt
New https://pypi.org/project/ultralytics/8.4.42 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.41  Python-3.10.5 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/Dataset KRSRI 2024 -2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov3-tinyu.pt, m

In [17]:
df = pd.DataFrame(RESULTS)

file_csv = "benchmark_results.csv"

# Jika file belum ada, tulis header.
# Jika sudah ada, append tanpa header agar kolom tidak dobel.
df.to_csv(
    file_csv,
    mode='a',
    header=not os.path.exists(file_csv),
    index=False
)

print("\nBenchmark selesai (data ditambahkan ke CSV).")
print(df)


Benchmark selesai (data ditambahkan ke CSV).
          Model     mAP50  mAP50-95  Precision    Recall    FPS_E2E  \
0  yolov3-tinyu  0.946072  0.666299   0.936962  0.945572  68.994353   
1  yolov6n.yaml  0.989381  0.957946   0.989718  0.992639  55.549869   
2      yolov10n  0.991364  0.970124   0.984606  0.985299  54.116511   

      FPS_Raw  Latency_ms    Size_MB  Training_Min  
0  106.461946   14.493940  23.231005     19.071886  
1   89.049496   18.001842   8.285536     49.191521  
2   62.537509   18.478649   5.486261     81.678155  


In [18]:
data_becnmark = pd.read_csv("benchmark_results.csv")

In [20]:
data_becnmark

,Model,mAP50,mAP50-95,Precision,Recall,FPS_E2E,FPS_Raw,Latency_ms,Size_MB,Training_Min
0,yolov5nu,0.991450,0.957035,0.986009,0.993721,57.320558,83.130386,17.445748,5.023134,30.140428
1,yolov8n,0.991557,0.973797,0.994079,0.995654,65.311929,92.522024,15.311139,5.960489,85.979723
2,yolov9s,0.989815,0.960026,0.987344,0.995296,26.748178,30.208653,37.385724,14.519967,94.503347
3,yolo11n,0.990103,0.958301,0.987080,0.993775,48.141841,72.662919,20.771952,5.215052,39.610919
4,yolo12n,0.989160,0.963693,0.994788,0.994536,34.202825,42.737510,29.237351,5.259607,55.589635
5,yolo26n,0.986403,0.955796,0.971831,0.947246,48.647469,58.603881,20.556054,5.134282,45.345216
6,yolov3-tinyu,0.946072,0.666299,0.936962,0.945572,68.994353,106.461946,14.493940,23.231005,19.071886
7,yolov6n.yaml,0.989381,0.957946,0.989718,0.992639,55.549869,89.049496,18.001842,8.285536,49.191521
8,yolov10n,0.991364,0.970124,0.984606,0.985299,54.116511,62.537509,18.478649,5.486261,81.678155
